# EDA: สำรวจผล BoT-IoT

Notebook นี้เป็นงานสำรวจหลัง `harmonization.ipynb`; ไม่เปลี่ยนข้อมูลและไม่ทำขั้น cleansing ซ้ำ. ตารางแสดงจำนวน flow, ป้ายกำกับ, protocol/service/state และค่าสถิติ feature.


## 1. หา output และตรวจตารางที่เชื่อมด้วย record_id

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATASET_NAME = 'BoT-IoT'
def find_dataset_dir(start: Path, dataset_name: str) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if candidate.name == dataset_name and (candidate / "download.py").is_file():
            return candidate
        nested = candidate / "data-pipeline" / "sub-dataset" / dataset_name
        if (nested / "download.py").is_file():
            return nested
    raise FileNotFoundError(f"หาโฟลเดอร์ {dataset_name} จาก {start} ไม่พบ")

DATASET_DIR = find_dataset_dir(Path.cwd(), DATASET_NAME)
HARM_DIR = DATASET_DIR / "output" / "harmonized"
features, labels, metadata = [
    pd.read_csv(HARM_DIR / f"{name}.csv", low_memory=False)
    for name in ["features", "labels", "metadata"]
]
aligned = all(frame["record_id"].is_unique for frame in [features, labels, metadata]) and features["record_id"].tolist() == labels["record_id"].tolist() == metadata["record_id"].tolist()
print("Rows:", len(features), "| IDs aligned:", aligned)
if not aligned:
    raise ValueError("record_id ไม่ตรงกันระหว่างตาราง")


## 2. สำรวจป้ายกำกับและแหล่ง split

In [ ]:
display(labels["attack_category"].value_counts(dropna=False).rename("rows").to_frame())
display(metadata["source_split"].value_counts(dropna=False).rename("rows").to_frame())


## 3. สำรวจชนิด traffic และขนาดการเชื่อมต่อ

In [ ]:
for column in ["protocol", "service_group", "conn_state_group"]:
    print(column)
    display(features[column].value_counts(dropna=False).rename("rows").to_frame())

numeric = [column for column in ["duration", "bytes_forward", "bytes_backward", "packets_forward", "packets_backward", "mean_packet_size"] if column in features]
display(features[numeric].describe(percentiles=[0.5, 0.9, 0.99]).T)


## 4. ตรวจรายงาน harmonization

ใช้รายงานนี้เพื่อดูขนาดตารางและ quality flags ก่อนส่ง CENTRAL builder.

In [ ]:
for filename in ["harmonization_report.csv", "harmonization_quality.csv"]:
    path = HARM_DIR / filename
    if path.is_file():
        print(filename)
        display(pd.read_csv(path))
